# Optimisation du Problème du Voyageur de Commerce
## Algorithme Génétique Multi-Objectif

**Objectif:** Trouver le circuit hamiltonien minimisant F = f1 + f2
- f1: Distance totale parcourue
- f2: Coût total du voyage

In [ ]:
import csv
import random
import os
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

## Configuration de l'Algorithme

In [ ]:
@dataclass
class ConfigGA:
    """Configuration de l'algorithme génétique."""
    taille_population: int = 300
    nb_generations: int = 500
    taux_croisement: float = 0.70
    taux_mutation: float = 0.15
    taille_tournoi: int = 5

config = ConfigGA()

## Chargement des Données

In [ ]:
class DonneesTSP:
    """Gestionnaire des données du problème TSP."""
    
    def __init__(self):
        self.villes: List[str] = []
        self.matrice_distances: List[List[int]] = []
        self.matrice_couts: List[List[int]] = []
    
    def charger_fichiers(self, fichier_distances: str, fichier_couts: str) -> None:
        """Charge les matrices depuis les fichiers CSV."""
        self.matrice_distances, self.villes = self._lire_matrice(fichier_distances)
        self.matrice_couts, _ = self._lire_matrice(fichier_couts)
        print(f"Chargement réussi: {len(self.villes)} villes")
    
    def _lire_matrice(self, chemin: str) -> Tuple[List[List[int]], List[str]]:
        """Lit une matrice depuis un fichier CSV."""
        matrice = []
        noms = []
        with open(chemin, 'r', encoding='utf-8') as f:
            lecteur = csv.reader(f)
            entete = next(lecteur)
            noms = entete[1:]
            for ligne in lecteur:
                if ligne:
                    matrice.append([int(v) for v in ligne[1:]])
        return matrice, noms
    
    def obtenir_distance(self, ville_a: str, ville_b: str) -> int:
        """Retourne la distance entre deux villes."""
        i = self.villes.index(ville_a)
        j = self.villes.index(ville_b)
        return self.matrice_distances[i][j]
    
    def obtenir_cout(self, ville_a: str, ville_b: str) -> int:
        """Retourne le coût entre deux villes."""
        i = self.villes.index(ville_a)
        j = self.villes.index(ville_b)
        return self.matrice_couts[i][j]


# Initialisation des données
donnees = DonneesTSP()
donnees.charger_fichiers("distances.csv", "cities.csv")
print(f"Villes: {donnees.villes}")

## Représentation d'une Solution (Chromosome)

In [ ]:
@dataclass
class Circuit:
    """Représente un circuit (solution candidate)."""
    parcours: List[str] = field(default_factory=list)
    distance_totale: int = 0
    cout_total: int = 0
    score: int = 0  # F = distance + cout
    
    @staticmethod
    def creer_aleatoire(liste_villes: List[str]) -> 'Circuit':
        """Génère un circuit aléatoire."""
        parcours = liste_villes.copy()
        random.shuffle(parcours)
        return Circuit(parcours=parcours)
    
    def calculer_score(self, donnees: DonneesTSP) -> None:
        """Évalue le circuit selon les deux objectifs."""
        self.distance_totale = 0
        self.cout_total = 0
        
        # Calculer pour chaque étape du parcours
        for idx in range(len(self.parcours) - 1):
            depart = self.parcours[idx]
            arrivee = self.parcours[idx + 1]
            self.distance_totale += donnees.obtenir_distance(depart, arrivee)
            self.cout_total += donnees.obtenir_cout(depart, arrivee)
        
        # Retour à la ville de départ
        derniere = self.parcours[-1]
        premiere = self.parcours[0]
        self.distance_totale += donnees.obtenir_distance(derniere, premiere)
        self.cout_total += donnees.obtenir_cout(derniere, premiere)
        
        # Score combiné
        self.score = self.distance_totale + self.cout_total
    
    def vers_chaine(self) -> str:
        """Convertit le parcours en chaîne pour export XML."""
        return ",".join(self.parcours + [self.parcours[0]])
    
    def copier(self) -> 'Circuit':
        """Crée une copie du circuit."""
        return Circuit(
            parcours=self.parcours.copy(),
            distance_totale=self.distance_totale,
            cout_total=self.cout_total,
            score=self.score
        )
    
    def __repr__(self) -> str:
        trajet = " → ".join(self.parcours + [self.parcours[0]])
        return f"F={self.score} (d={self.distance_totale}, c={self.cout_total}) | {trajet}"

## Opérateurs Génétiques

In [ ]:
class OperateursGenetiques:
    """Implémente les opérateurs de croisement et mutation."""
    
    @staticmethod
    def croisement_deux_points(parent1: Circuit, parent2: Circuit, toutes_villes: List[str]) -> Circuit:
        """
        Croisement à deux points.
        Copie un segment du parent1, complète avec parent2.
        Remplace les doublons par les villes manquantes.
        """
        taille = len(parent1.parcours)
        
        # Choisir deux points de coupure
        pt1 = random.randint(1, taille - 2)
        pt2 = random.randint(pt1 + 1, taille - 1)
        
        # Commencer avec le parcours du parent1
        nouveau_parcours = parent1.parcours.copy()
        
        # Insérer le segment du parent2
        for i in range(pt1, pt2):
            nouveau_parcours[i] = parent2.parcours[i]
        
        # Identifier et corriger les doublons
        villes_vues = set()
        positions_doublons = []
        
        for pos, ville in enumerate(nouveau_parcours):
            if ville in villes_vues:
                positions_doublons.append(pos)
            else:
                villes_vues.add(ville)
        
        # Trouver les villes manquantes
        villes_manquantes = list(set(toutes_villes) - villes_vues)
        random.shuffle(villes_manquantes)
        
        # Remplacer les doublons
        for idx, pos in enumerate(positions_doublons):
            nouveau_parcours[pos] = villes_manquantes[idx]
        
        return Circuit(parcours=nouveau_parcours)
    
    @staticmethod
    def mutation_echange(circuit: Circuit, taux: float) -> None:
        """
        Mutation par échange de deux villes.
        Modifie le circuit en place.
        """
        if random.random() < taux:
            taille = len(circuit.parcours)
            if taille >= 2:
                i, j = random.sample(range(taille), 2)
                circuit.parcours[i], circuit.parcours[j] = circuit.parcours[j], circuit.parcours[i]

## Moteur de l'Algorithme Génétique

In [ ]:
class MoteurGA:
    """Moteur principal de l'algorithme génétique."""
    
    def __init__(self, donnees: DonneesTSP, config: ConfigGA):
        self.donnees = donnees
        self.config = config
        self.population: List[Circuit] = []
        self.meilleur_global: Optional[Circuit] = None
        self.generation_meilleur: int = 0
    
    def initialiser_population(self) -> None:
        """Crée la population initiale aléatoire."""
        self.population = [
            Circuit.creer_aleatoire(self.donnees.villes)
            for _ in range(self.config.taille_population)
        ]
    
    def evaluer_population(self) -> None:
        """Calcule le score de tous les individus."""
        for circuit in self.population:
            circuit.calculer_score(self.donnees)
    
    def trier_population(self) -> None:
        """Trie la population par score croissant (meilleur = plus petit)."""
        self.population.sort(key=lambda c: c.score)
    
    def selection_tournoi(self) -> Circuit:
        """Sélectionne un individu par tournoi."""
        participants = random.sample(self.population, self.config.taille_tournoi)
        participants.sort(key=lambda c: c.score)
        return participants[0]
    
    def creer_nouvelle_generation(self) -> List[Circuit]:
        """Génère la prochaine génération."""
        nouvelle_pop = []
        
        # Élitisme: conserver le meilleur
        nouvelle_pop.append(self.population[0].copier())
        
        # Remplir le reste de la population
        while len(nouvelle_pop) < self.config.taille_population:
            if random.random() < self.config.taux_croisement:
                # Croisement
                parent_a = self.selection_tournoi()
                parent_b = self.selection_tournoi()
                enfant = OperateursGenetiques.croisement_deux_points(
                    parent_a, parent_b, self.donnees.villes
                )
            else:
                # Copie simple
                enfant = self.selection_tournoi().copier()
            
            # Mutation
            OperateursGenetiques.mutation_echange(enfant, self.config.taux_mutation)
            nouvelle_pop.append(enfant)
        
        return nouvelle_pop
    
    def executer(self) -> Circuit:
        """Lance l'algorithme génétique."""
        print(f"Démarrage - Population: {self.config.taille_population}, Générations: {self.config.nb_generations}")
        print("=" * 70)
        
        self.initialiser_population()
        
        for gen in range(self.config.nb_generations):
            self.evaluer_population()
            self.trier_population()
            
            champion = self.population[0]
            
            # Mettre à jour le meilleur global
            if self.meilleur_global is None or champion.score < self.meilleur_global.score:
                self.meilleur_global = champion.copier()
                self.generation_meilleur = gen
                print(f"Gén {gen:4d}: F={champion.score} (dist={champion.distance_totale}, coût={champion.cout_total})")
            
            # Créer la génération suivante
            self.population = self.creer_nouvelle_generation()
        
        print("=" * 70)
        print(f"\nMeilleure solution trouvée à la génération {self.generation_meilleur}:")
        print(self.meilleur_global)
        
        return self.meilleur_global

## Gestion des Solutions (Sauvegarde XML)

In [ ]:
class GestionnaireSolutions:
    """Gère la sauvegarde et le chargement des solutions."""
    
    def __init__(self, fichier: str = "solution.xml"):
        self.fichier = fichier
        self.meilleur_score: int = -1
        self.solutions: List[str] = []
    
    def charger(self) -> None:
        """Charge les solutions existantes depuis le fichier XML."""
        if not os.path.exists(self.fichier):
            return
        
        try:
            arbre = ET.parse(self.fichier)
            racine = arbre.getroot()
            
            elem_fitness = racine.find('BestFitness')
            if elem_fitness is not None:
                self.meilleur_score = int(elem_fitness.text)
            else:
                # Format ancien
                elem_fitness = racine.find('.//Fitness')
                if elem_fitness is not None:
                    self.meilleur_score = int(elem_fitness.text)
            
            self.solutions = [elem.text for elem in racine.findall('.//Tour')]
        except Exception as e:
            print(f"Erreur de chargement: {e}")
    
    def sauvegarder(self, score: int, distance: int, cout: int, tours: List[str]) -> None:
        """Sauvegarde les solutions dans le fichier XML."""
        racine = ET.Element('TSP_Solutions')
        
        ET.SubElement(racine, 'BestFitness').text = str(score)
        ET.SubElement(racine, 'TotalSolutions').text = str(len(tours))
        
        for idx, tour in enumerate(tours, 1):
            solution = ET.SubElement(racine, 'Solution', id=str(idx))
            ET.SubElement(solution, 'Tour').text = tour
            ET.SubElement(solution, 'Distance').text = str(distance)
            ET.SubElement(solution, 'Cost').text = str(cout)
        
        arbre = ET.ElementTree(racine)
        ET.indent(arbre, space="    ")
        arbre.write(self.fichier, encoding='utf-8', xml_declaration=True)
        
        self.meilleur_score = score
        self.solutions = tours
    
    def traiter_resultat(self, circuit: Circuit) -> None:
        """Traite le résultat et met à jour si nécessaire."""
        nouveau_tour = circuit.vers_chaine()
        
        print("\n" + "=" * 70)
        
        if self.meilleur_score < 0:
            # Première solution
            self.sauvegarder(circuit.score, circuit.distance_totale, circuit.cout_total, [nouveau_tour])
            print("PREMIÈRE SOLUTION ENREGISTRÉE")
            print(f"Score: {circuit.score}")
        
        elif circuit.score < self.meilleur_score:
            # Nouvelle meilleure solution
            self.sauvegarder(circuit.score, circuit.distance_totale, circuit.cout_total, [nouveau_tour])
            print(f"NOUVEAU RECORD! ({self.meilleur_score} → {circuit.score})")
        
        elif circuit.score == self.meilleur_score:
            # Même score, vérifier si nouveau parcours
            if nouveau_tour not in self.solutions:
                self.solutions.append(nouveau_tour)
                self.sauvegarder(circuit.score, circuit.distance_totale, circuit.cout_total, self.solutions)
                print(f"NOUVEAU PARCOURS DÉCOUVERT (même score: {circuit.score})")
                print(f"Total de parcours optimaux: {len(self.solutions)}")
            else:
                print(f"PARCOURS DÉJÀ CONNU (score: {circuit.score})")
        
        else:
            # Moins bon
            print(f"PAS D'AMÉLIORATION (actuel: {circuit.score}, meilleur: {self.meilleur_score})")
            print(f"Conservation des {len(self.solutions)} solution(s) existante(s)")
        
        print("=" * 70)

## Exécution de l'Algorithme

In [ ]:
# Charger les solutions précédentes
gestionnaire = GestionnaireSolutions("solution.xml")
gestionnaire.charger()

if gestionnaire.meilleur_score > 0:
    print(f"Meilleur score précédent: {gestionnaire.meilleur_score}")
    print(f"Nombre de solutions: {len(gestionnaire.solutions)}")
    print()

# Lancer l'algorithme
moteur = MoteurGA(donnees, config)
resultat = moteur.executer()

# Traiter et sauvegarder si amélioration
gestionnaire.traiter_resultat(resultat)

## Afficher Toutes les Solutions Optimales

In [ ]:
# Recharger et afficher
gestionnaire.charger()

print(f"Meilleur score (F): {gestionnaire.meilleur_score}")
print(f"Nombre de parcours optimaux: {len(gestionnaire.solutions)}")
print()

for i, tour in enumerate(gestionnaire.solutions, 1):
    print(f"Parcours {i}: {tour}")

## Exécution Multiple (Recherche Intensive)

In [ ]:
def recherche_intensive(nb_executions: int = 10) -> None:
    """Lance plusieurs exécutions pour trouver plus de solutions."""
    print(f"Lancement de {nb_executions} exécutions...\n")
    
    for i in range(nb_executions):
        print(f"\n{'='*70}")
        print(f"EXÉCUTION {i+1}/{nb_executions}")
        print(f"{'='*70}")
        
        moteur = MoteurGA(donnees, config)
        resultat = moteur.executer()
        
        gestionnaire.charger()  # Recharger l'état actuel
        gestionnaire.traiter_resultat(resultat)
    
    # Résumé final
    gestionnaire.charger()
    print(f"\n{'#'*70}")
    print(f"RÉSUMÉ FINAL")
    print(f"{'#'*70}")
    print(f"Meilleur score: {gestionnaire.meilleur_score}")
    print(f"Parcours trouvés: {len(gestionnaire.solutions)}")
    print(f"{'#'*70}")

# Décommenter pour lancer une recherche intensive:
# recherche_intensive(10)